In [1]:
from tts_utilities.jnb_util import this_notebook_path
from tts_utilities.util import as_list
from demosat_data_utils.ephemeris import EphemerisContainer
from demosat_data_utils.comm_windows import CommWindowContainer
from demosat_data_utils.orbit_events import OrbitEventContainer
from demosat_data_utils.ground_stations import GroundStationContainer
from demosat_plan.demosat_scheduler import DemosatScheduler
from demosat_seq.background_seq_builder import DemoSatBackgroundSeqBuilder
from demosat_seq.sequence_delivery_manager import DemoSatSequenceDeliveryManager
import demosat_dict
from pathlib import Path
import shutil

2026-025T07:09:36 INFO     2026-025T07:09:36 (INFO) tts_spice.furnish.furnish_kernel: Loaded SPICE   ]8;id=814398;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_spice/src/tts_spice/furnish.py\furnish.py]8;;\:]8;id=112537;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_spice/src/tts_spice/furnish.py#158\158]8;;\
                           kernel:                                                                                 
                           /Users/muszynsk/projects/tt_studio/dev/tts_core/tts_spice/src/tts_spice/k               
                           ernels/naif0012.tls                                                                     

                  INFO     2026-025T07:09:36 (INFO) tts_spice.furnish.furnish_kernel: Loaded SPICE   ]8;id=387829;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_spice/src/tts_spice/furnish.py\furnish.py]8;;\:]8;id=722443;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_spice/src/tts_spice/furnish.py#158\158]8;;\
                           kernel:                                                                                 
                           /Users/muszynsk/projects/tt_studio/dev/tts_core/tts_spice/src/tts_spice/k               
                           ernels/de430.bsp                                                                        

                  INFO     2026-025T07:09:36 (INFO) tts_spice.furnish.furnish_kernel: Loaded SPICE   ]8;id=979084;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_spice/src/tts_spice/furnish.py\furnish.py]8;;\:]8;id=540006;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_spice/src/tts_spice/furnish.py#158\158]8;;\
                           kernel:                                                                                 
                           /Users/muszynsk/projects/tt_studio/dev/tts_core/tts_spice/src/tts_spice/k               
                           ernels/pck00010.tpc                                                                     

                  INFO     2026-025T07:09:36 (INFO) tts_spice.furnish.furnish_kernel: Loaded SPICE   ]8;id=195083;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_spice/src/tts_spice/furnish.py\furnish.py]8;;\:]8;id=354433;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_spice/src/tts_spice/furnish.py#158\158]8;;\
                           kernel:                                                                                 
                           /Users/muszynsk/projects/tt_studio/dev/tts_core/tts_spice/src/tts_spice/k               
                           ernels/earth_latest_high_prec.bpc                                                       

# Clean Up Previous Runs

For this demo only, we first remove any sequences from the 2024_02_02 planning cycle. This is done in case the user has run this demo previously as it is meant to run cleanly from the start. This is not flightlike. We also copy the directory `seed_seq_db` which is meant to represent the starting point of the demo. It should not be edited directly. `seed_seq_db` is a resonably exectation for what the seq_db might look like at the beginning of the 2024_02_02 planning cycle (although in reality there would be many more past cycles there. One is enough for a demo).

In [2]:
target_dir = this_notebook_path().parent.joinpath(f'seq_db')
if target_dir.exists() and target_dir.is_dir():
    shutil.rmtree(target_dir)
    
source = this_notebook_path().parent.joinpath(f'seed_seq_db')
dest = this_notebook_path().parent.joinpath(f'seq_db')
shutil.copytree(source, dest)

PosixPath('/Users/muszynsk/projects/tt_studio/dev/demosat/demosat_seq/examples/seq_db')

# Build schedule
This step repeats the steps already demonstrated in demosat_plan's example jnb. On a real project, the plan JSON would have been delivered to a location where this process could pick it up, but since we want this example to be able to be run independently of tts_plan's examples, we just redo it minimally here.

In [3]:
ephemeris_path = this_notebook_path().parent.joinpath("ephemeris.txt")
ephem_container = EphemerisContainer(csv_path=ephemeris_path, cast_fields=True)
orbit_events = OrbitEventContainer(ephem=ephem_container)
start_time = min(item.time for item in ephem_container)
stations = GroundStationContainer()
min_elevation = 30
comm_windows = CommWindowContainer(ephem=ephem_container, stations=stations, min_el=min_elevation)

end_time = max(item.time for item in ephem_container)
scheduler = DemosatScheduler(
    start_time=start_time,
    end_time=end_time,
    ephemeris_container=ephem_container,
    comm_window_container=comm_windows,
    orbit_event_container=orbit_events,
)
scheduler.build_schedule()
scheduler.to_json_file('demosat_schedule_2024_02_02.json')

2026-025T07:09:37 INFO     2026-025T07:09:37 (INFO) tts_plan.scheduler.to_json_file: Scheduler     ]8;id=793757;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_plan/src/tts_plan/scheduler.py\scheduler.py]8;;\:]8;id=320052;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_plan/src/tts_plan/scheduler.py#390\390]8;;\
                           data saved to demosat_schedule_2024_02_02.json                                          

In [4]:
dict_pkg_dir = Path(demosat_dict.__file__).parent
cmd_dict_path = dict_pkg_dir.joinpath('dictionaries/v1/command.xml')
background_seq_builder = DemoSatBackgroundSeqBuilder('demosat_schedule_2024_02_02.json', cmd_dict_path, 'background_2024_02_02')
background_sequence = background_seq_builder.build_sequence()

In [5]:
sdm = DemoSatSequenceDeliveryManager('2024_02_02', this_notebook_path().parent.joinpath(f'seq_db'), background_sequence, cmd_dict_path)
sdm.create_delivery_directories()

In [6]:
sdm.collect_descendants()

2026-025T07:09:37 INFO     2026-025T07:09:37 (INFO)                                ]8;id=150886;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_seq/src/tts_seq/core/sequence_delivery_manager.py\sequence_delivery_manager.py]8;;\:]8;id=180893;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_seq/src/tts_seq/core/sequence_delivery_manager.py#108\108]8;;\
                           tts_seq.core.sequence_delivery_manager.collect_descenda                                 
                           nts: Collected 5 descendant sequences                                                   

In [7]:
sdm.approve_sequences()

                  INFO     2026-025T07:09:37 (INFO)                                ]8;id=491731;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_seq/src/tts_seq/core/sequence_delivery_manager.py\sequence_delivery_manager.py]8;;\:]8;id=406313;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_seq/src/tts_seq/core/sequence_delivery_manager.py#138\138]8;;\
                           tts_seq.core.sequence_delivery_manager.approve_sequence                                 
                           s: Approved sequence mco_sband_setup written to                                         
                           /Users/muszynsk/projects/tt_studio/dev/demosat/demosat_                                 
                           seq/examples/seq_db/2024_02_02/approved/mco_sband_setup                                 
                           .seq.json                                                                               

                  INFO     2026-025T07:09:37 (INFO)                                ]8;id=627659;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_seq/src/tts_seq/core/sequence_delivery_manager.py\sequence_delivery_manager.py]8;;\:]8;id=8581;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_seq/src/tts_seq/core/sequence_delivery_manager.py#138\138]8;;\
                           tts_seq.core.sequence_delivery_manager.approve_sequence                                 
                           s: Approved sequence break_down_comm written to                                         
                           /Users/muszynsk/projects/tt_studio/dev/demosat/demosat_                                 
                           seq/examples/seq_db/2024_02_02/approved/break_down_comm                                 
                           .seq.json                                                                               

                  INFO     2026-025T07:09:37 (INFO)                                ]8;id=740030;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_seq/src/tts_seq/core/sequence_delivery_manager.py\sequence_delivery_manager.py]8;;\:]8;id=63592;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_seq/src/tts_seq/core/sequence_delivery_manager.py#138\138]8;;\
                           tts_seq.core.sequence_delivery_manager.approve_sequence                                 
                           s: Approved sequence asf_sband_setup written to                                         
                           /Users/muszynsk/projects/tt_studio/dev/demosat/demosat_                                 
                           seq/examples/seq_db/2024_02_02/approved/asf_sband_setup                                 
                           .seq.json                                                                               

                  INFO     2026-025T07:09:37 (INFO)                                ]8;id=654895;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_seq/src/tts_seq/core/sequence_delivery_manager.py\sequence_delivery_manager.py]8;;\:]8;id=826243;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_seq/src/tts_seq/core/sequence_delivery_manager.py#138\138]8;;\
                           tts_seq.core.sequence_delivery_manager.approve_sequence                                 
                           s: Approved sequence dark_side_calibration written to                                   
                           /Users/muszynsk/projects/tt_studio/dev/demosat/demosat_                                 
                           seq/examples/seq_db/2024_02_02/approved/dark_side_calib                                 
                           ration.seq.json                                                                         

                  INFO     2026-025T07:09:37 (INFO)                                ]8;id=544211;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_seq/src/tts_seq/core/sequence_delivery_manager.py\sequence_delivery_manager.py]8;;\:]8;id=429930;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_seq/src/tts_seq/core/sequence_delivery_manager.py#138\138]8;;\
                           tts_seq.core.sequence_delivery_manager.approve_sequence                                 
                           s: Approved sequence wgs_sband_setup written to                                         
                           /Users/muszynsk/projects/tt_studio/dev/demosat/demosat_                                 
                           seq/examples/seq_db/2024_02_02/approved/wgs_sband_setup                                 
                           .seq.json                                                                               

                  INFO     2026-025T07:09:37 (INFO)                                ]8;id=49062;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_seq/src/tts_seq/core/sequence_delivery_manager.py\sequence_delivery_manager.py]8;;\:]8;id=230423;file:///Users/muszynsk/projects/tt_studio/dev/tts_core/tts_seq/src/tts_seq/core/sequence_delivery_manager.py#138\138]8;;\
                           tts_seq.core.sequence_delivery_manager.approve_sequence                                 
                           s: Approved sequence background_2024_02_02 written to                                   
                           /Users/muszynsk/projects/tt_studio/dev/demosat/demosat_                                 
                           seq/examples/seq_db/2024_02_02/approved/background_2024                                 
                           _02_02.seq.json                                                                         

In [8]:
print(sdm.descendant_seqdicts[0].to_dotseq())

;on_board_filename=mco_sband_setup
;planId=1
;timeSorted=false
;simulationDatasetId=1
;
;
; This is a demonstration of how tts_seq_viewer can work for a simple earth orbiting mission.
; The details of this sequence don't matter much, but for context, it is meant to represent
; a simple spacecraft setting up comm for an X-Band contact with Wallops ground station.
;
; This is a realtively timed sequence because it could be used at any given time depending on
; the exact geometries and view periods to Wallops.
;
; In a real sequence, one might expect metatdata to also go here.
;------------------------------------------------------------------------------------------------

R00:00:01 TX_POWER ON,XBAND
R00:00:05 CLOSE_SCI_PRODUCTS
R00:00:05 FLUSH_DL_BUFFERS
R00:00:01 SET_VCDU 17
R00:00:01 SET_TLM_PROD_RATE XBAND_DL
R00:00:01 SET_PLAYBACK_RATE 200_MBPS
R00:00:01 START_PLAYBACK



In [9]:
sdm.command_dictionary_path

PosixPath('/Users/muszynsk/projects/tt_studio/dev/demosat/demosat_dict/src/demosat_dict/dictionaries/v1/command.xml')

In [10]:
from demosat_dictionary_interface.command import CommandDictionary

In [11]:
cmd = CommandDictionary(sdm.command_dictionary_path)

In [12]:
for x in cmd: print(x.stem)


TEST_HW_CMD_1
TEST_HW_CMD_2
RUN_SEQ
DO_SCIENCE
STOP_SCIENCE
DARK_SIDE_CALIBRATION
ADCS_YAW
TX_POWER
SET_VCDU
CLOSE_SCI_PRODUCTS
FLUSH_DL_BUFFERS
SET_TLM_PROD_RATE
SET_PLAYBACK_RATE
START_PLAYBACK
STOP_PLAYBACK
SET_HEATER_DUTY
UPDATE_CONFIG_KEY
UPLOAD_FILTER_TABLE
FORCE_HIGH_SPEED_DUMP
SET_GUIDANCE_MODE
TRACK_GROUND_TARGET
TRACK_INERTIAL_TARGET
CONFIG_INSTRUMENT
TAKE_IMAGE
UPLOAD_EPHEMERIS


In [13]:
for a in x.args: print(a.default)


None
